## Lab: Multi-variable neural-network regression

In this lab we perform multi-variable regression with a dense feed forward ANN on the Boston housing data-set using Keras.
  
`Note`: It is recommended that you do this lab with `Google Co-lab`, so that you don’t have to install Tensor-flow on your laptop.

**Submission:**

* You need to upload ONE document to Canvas when you are done
  * (1) A PDF (or HTML) of the completed form of this notebook
* The final uploaded version should NOT have any code-errors present
* All outputs must be visible in the uploaded version, including code-cell outputs, images, graphs, etc



**Instructions**

* **You only need to do this week's example with Keras (i.e. no PyTorch)**
  * `Pre-process`
    * Normalize the data as needed
    * Partition data into training, validation, and test
  * `Model` and `compilation`
    * Code regression using a deep feed forward fully connected Neural network
    * Use the correct loss function and output layer activation for this learning task
    * Set up your model, use a dense feed forward ANN model with at least two hidden layers
    * Make sure you have it set to regularize and do early stopping
  * `Tuning`
    * Do `MANUAL` hyper parameter tuning to try to achieve an optimal fit model
    * Try 2 or 3 model configurations (show the training/validation errors for each configuration)
      * just explore with trial and error
    * You `MUST` use early stopping: [click here](https://keras.io/api/callbacks/early_stopping/)
    * Explore L1 and L2 regularization or dropout
    * Explore different optimizers
    * Explore different options for activation functions, network size/depth, etc
  * `Final results`
    * Visualize & report the results at the end  
    * Monitor training and validation throughout training by plotting
    * (convergence plots, parity plots, report final training/validation/test errors)

* **Document what is going on in the code, as needed, with narrative markdown text between cells.**
* **Reference: This assignment is similar an example in the text book**
  * See the textbook (Chollet (first edition) chapter-3, page 85) for reference

## Data

In [2]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [3]:
from tensorflow.keras.datasets import boston_housing

# note: the testing data gets partitioned later into validation in the k-fold step
(train_data, train_targets), (test_data, test_targets) = (
    boston_housing.load_data())

57026/57026 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
print("Training data shape:", train_data.shape)
print("Training targets shape:", train_targets.shape)
print("Test data shape:", test_data.shape)
print("Test targets shape:", test_targets.shape)

Training data shape: (404, 13)
Training targets shape: (404,)
Test data shape: (102, 13)
Test targets shape: (102,)


A glimpse of the first 5 rows of the training data and targets:

In [9]:
print("First 5 rows of training data:\n", train_data[:5])
print("First 5 training targets:\n", train_targets[:5])

First 5 rows of training data:
 [[1.23247e+00 0.00000e+00 8.14000e+00 0.00000e+00 5.38000e-01 6.14200e+00
  9.17000e+01 3.97690e+00 4.00000e+00 3.07000e+02 2.10000e+01 3.96900e+02
  1.87200e+01]
 [2.17700e-02 8.25000e+01 2.03000e+00 0.00000e+00 4.15000e-01 7.61000e+00
  1.57000e+01 6.27000e+00 2.00000e+00 3.48000e+02 1.47000e+01 3.95380e+02
  3.11000e+00]
 [4.89822e+00 0.00000e+00 1.81000e+01 0.00000e+00 6.31000e-01 4.97000e+00
  1.00000e+02 1.33250e+00 2.40000e+01 6.66000e+02 2.02000e+01 3.75520e+02
  3.26000e+00]
 [3.96100e-02 0.00000e+00 5.19000e+00 0.00000e+00 5.15000e-01 6.03700e+00
  3.45000e+01 5.98530e+00 5.00000e+00 2.24000e+02 2.02000e+01 3.96900e+02
  8.01000e+00]
 [3.69311e+00 0.00000e+00 1.81000e+01 0.00000e+00 7.13000e-01 6.37600e+00
  8.84000e+01 2.56710e+00 2.40000e+01 6.66000e+02 2.02000e+01 3.91430e+02
  1.46500e+01]]
First 5 training targets:
 [15.2 42.3 50.  21.1 17.7]


## Pre-processing

### Data Normalization

Normalize the input features so that they all have similar scales.

In [5]:
mean = train_data.mean(axis=0)
std = train_data.std(axis=0)
train_data = (train_data - mean)/std
test_data = (test_data - mean)/std

### Model Compilation

Define a deep feed-forward neural network with two hidden layers, using `relu` activation for the hidden layers and a `linear` activation for the output layer. `Adam` is the optimizer. `L2 regularization` will be applied. `Mean Squared Error (MSE)` will be used as training loss function.

In [24]:
from tensorflow.keras import models, layers, regularizers

# define the ANN model with l2 as regularizer; num_nodes as layer size; adam as optimizer; relu as activation. These paramters will be explored in hyperparameter tunning later.
def build_model(input_shape, num_nodes, l2_reg=0.001, optimizer='adam', activation='relu'):
    # construct neural network model
    model = models.Sequential()
    # add first layer with nodes and activation
    model.add(layers.Dense(num_nodes, activation=activation, input_shape=(input_shape,),
                           kernel_regularizer=regularizers.l2(l2_reg)))
    # add second layer with nodes and activation
    model.add(layers.Dense(num_nodes, activation=activation,
                           kernel_regularizer=regularizers.l2(l2_reg)))
    # add output layer with a single unit and no activation
    model.add(layers.Dense(1))
    # compile model with optimizer and training loss function
    model.compile(optimizer=optimizer, loss='mse')
    return model


### View model

In [25]:
# define the parameters values
input_shape = train_data.shape[1] # number of input feature
num_nodes = 64 # layer size

model = build_model(input_shape, num_nodes)
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,121 (20.00 KB)

 Trainable params: 5,121 (20.00 KB)

 Non-trainable params: 0 (0.00 B)

### Early Stopping

Build early stopping by using `val_mse` as monitor metric and stops training if the metric stops improving for a certain number of epochs defined in `patience`

In [ ]:
from tensorflow.keras import callbacks

# build early stopping
early_stopping = callbacks.EarlyStopping(
    monitor='val_mse', # monitor validation mse
    patience=10,       # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True # restore model weights from the epoch with the best value
)

### Function to return and plot model results
Build train_test_evaluate_model to return train/test errors and plots

In [26]:
import matplotlib.pyplot as plt

def train_and_evaluate_model(input_shape, l2_reg_val, num_nodes, num_epochs=100, optimizer_name='adam', activation_func='relu', model_name="Model"):
    print(f"\n--- Training model with (L2 Regularization: {l2_reg_val}, Nodes: {num_nodes}, Optimizer: {optimizer_name}, Activation: {activation_func}) ---")

    # build the model with the hyperparemters: L2 regularization, number of nodes, optimizer, and activation
    model = build_model(input_shape, num_nodes=num_nodes, l2_reg=l2_reg_val, optimizer=optimizer_name, activation=activation_func)

    # train the model using early_stopping built above, and using test_data for validation
    history = model.fit(train_data, train_targets,
                        epochs=num_epochs, # num_epochs = 200
                        validation_data=(test_data, test_targets), # use test data as validation
                        callbacks=[early_stopping],  # early stopping is built in previous code
                        verbose=1) # set to 1 to monitor the progress of the epochs

    # record the final test mse when training stops
    test_mse = history.history['val_loss'][-1]
    print(f"Test MSE: {test_mse:.4f}")

    # output training and test losses (mse) for epochs
    train_loss = history.history['loss']
    test_loss = history.history['val_loss']
    epochs = range(1, len(train_loss) + 1)

    # plot training and test mse
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_loss, 'bo', label='Training MSE')
    plt.plot(epochs, test_loss, 'b', label='Test MSE')
    plt.title(f'Training and Test MSE')
    plt.xlabel('Epochs')
    plt.ylabel('MSE')
    plt.legend()
    plt.grid(True)
    plt.show()

    return history, model



### Tunnining
Instead of manual, here we are using a random search of parameters grid

In [ ]:
import random

input_shape = train_data.shape[1]

# create hyperparameters grid
l2_reg_options = [0.0001, 0.001, 0.01, 0.1, 1]
num_nodes_options = [16, 32, 64, 128]
optimizer_options = ['adam', 'sgd', 'rmsprop']
activation_options = ['relu', 'tanh', 'sigmoid', 'elu']

# number of random search iterations
num_random_trials = 3

results = []

for i in range(num_random_trials):
    # randomly sample hyperparameters for each trial
    random_l2_reg = random.choice(l2_reg_options)
    random_num_nodes = random.choice(num_nodes_options)
    random_optimizer = random.choice(optimizer_options)
    random_activation = random.choice(activation_options)

    model_name = f"Random Config {i+1}"

    # train and evaluate the model with the selected hyperparameters
    history, model = train_and_evaluate_model(
        input_shape,
        l2_reg_val=random_l2_reg,
        num_nodes=random_num_nodes,
        optimizer_name=random_optimizer,
        activation_func=random_activation,
        model_name=model_name
    )

    # store results
    results.append({
        'config_name': model_name,
        'l2_reg': random_l2_reg,
        'num_nodes': random_num_nodes,
        'optimizer': random_optimizer,
        'activation': random_activation,
        'history': history,
        'model': model
    })

# Example of how to access a result:
print(f"\nFirst configuration's final test MSE: {results[0]['history'].history['val_loss'][-1]:.4f}")

## Final results

## Generate HTML

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
import os

os.getcwd()

'/content/drive/MyDrive/GTU'

In [21]:
os.chdir('/content/drive/MyDrive/GTU/DSAN5300/labs/lab-W10-Keras-OPTIONAL-BONUS')

In [28]:
notebook_name = 'lab.ipynb' # Replace with your actual notebook name if different

# Construct the nbconvert command
command = f'jupyter nbconvert --to html {notebook_name}'

# Execute the command
os.system(command)

0